In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

CONVERTS_DIR = Path(r"C:/Users/STREAM/Desktop/order tracking/converts")

# ── Load source files ──────────────────────────────────────────────
df_sayim = pd.read_excel(CONVERTS_DIR / "mikro sayim.xlsx", header=0)
df_mikro = pd.read_excel(CONVERTS_DIR / "mikro.xlsx", header=0)
df_migration_example = pd.read_excel(CONVERTS_DIR / "MIGRATION_EXAMPLE (1).xlsx", header=0)

print(f"mikro sayim  : {df_sayim.shape[0]} rows x {df_sayim.shape[1]} cols")
print(f"mikro        : {df_mikro.shape[0]} rows x {df_mikro.shape[1]} cols")
print(f"migration tpl: {df_migration_example.shape[0]} rows x {df_migration_example.shape[1]} cols")
print(f"\nMigration columns: {df_migration_example.columns.tolist()}")

mikro sayim  : 71 rows x 13 cols
mikro        : 1244 rows x 16 cols
migration tpl: 3 rows x 13 cols

Migration columns: ['Malzeme Kodu', 'Malzeme Adı', 'Kategori', 'Departman', 'Birim', 'Min Stok', 'İdeal Stok Seviyesi (3 aylık)', 'Maksimum Stok Seviyesi', 'Mevcut Stok', 'Lot No', 'Son Kullanma', 'Marka', 'Tedarikçi']


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# NORMALIZE KEYS AND PREPARE DATA
# ══════════════════════════════════════════════════════════════════════

# Normalize join keys
df_sayim["_key"] = df_sayim["Katolog Numarası"].astype(str).str.strip()
df_mikro["_key"] = df_mikro["Malzeme Kodu"].astype(str).str.strip()

# Filter out IPTAL lots
df_mikro = df_mikro[df_mikro["Lot No"] != "IPTAL"].copy()

# Convert numeric and date columns
df_mikro["Gelen_Numeric"] = pd.to_numeric(df_mikro["Gelen Miktar"], errors="coerce").fillna(0)
df_mikro["Bittiği Tarih"] = pd.to_datetime(df_mikro["Bittiği Tarih"], errors="coerce")
df_mikro["Son Kullanma Tarihi"] = pd.to_datetime(df_mikro["Son Kullanma Tarihi"], errors="coerce")
df_mikro["Geliş Tarihi"] = pd.to_datetime(df_mikro["Geliş Tarihi"], errors="coerce")

print(f"mikro after filtering IPTAL: {len(df_mikro)} rows")

# ── Aggregate by unique (item, lot) ────────────────────────────────
# Sum Gelen Miktar per lot, take latest dates
lot_agg = df_mikro.groupby(["_key", "Lot No"]).agg({
    "Gelen_Numeric": "sum",
    "Bittiği Tarih": "max",      # if any row has finish date, lot is finished
    "Son Kullanma Tarihi": "max",
    "Geliş Tarihi": "max",
    "Dağıtımcı Firma": "first",
    "Malzeme Tanımı": "first",
    "department": "first",
}).reset_index()

lot_agg.columns = ["_key", "Lot_No", "Total_Received", "Finished_Date", "Expiry", "Received_Date", "Supplier", "Item_Name", "Department"]

# Mark active vs depleted
lot_agg["is_active"] = lot_agg["Finished_Date"].isna()
lot_agg["Status"] = lot_agg["is_active"].map({True: "ACTIVE", False: "DEPLETED"})

# currentQuantity: active lots keep their received qty, depleted lots = 0
lot_agg["currentQuantity"] = np.where(lot_agg["is_active"], lot_agg["Total_Received"], 0)

print(f"Unique lots: {len(lot_agg)}")
print(f"  ACTIVE (no Bittiği Tarih): {lot_agg['is_active'].sum()}")
print(f"  DEPLETED (has Bittiği Tarih): {(~lot_agg['is_active']).sum()}")

mikro after dedup (unique item+lot): 169 rows
Common codes:     41
Sayım-only codes: 29
Mikro-only codes: 20


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# VERIFICATION: Compare active lots sum vs sayım Depo
# ══════════════════════════════════════════════════════════════════════

# Sum currentQuantity per item from lots
lots_sum = lot_agg.groupby("_key")["currentQuantity"].sum().reset_index()
lots_sum.columns = ["_key", "Lots_Total_Stock"]

# Get sayım Depo
sayim_depo = df_sayim[["_key", " Depo"]].drop_duplicates("_key")
sayim_depo.columns = ["_key", "Sayim_Depo"]

# Merge for comparison
verify = lots_sum.merge(sayim_depo, on="_key", how="outer")
verify["Diff"] = verify["Lots_Total_Stock"].fillna(0) - verify["Sayim_Depo"].fillna(0)

# Identify code sets
sayim_codes = set(df_sayim["_key"])
mikro_codes = set(lot_agg["_key"])
common_codes = sayim_codes & mikro_codes
sayim_only_codes = sayim_codes - mikro_codes
mikro_only_codes = mikro_codes - sayim_codes

print("═" * 70)
print("VERIFICATION: Active lots sum vs Sayım Depo")
print("═" * 70)
print(f"\nItems in both files: {len(common_codes)}")
print(f"Items only in sayım (no lot data): {len(sayim_only_codes)}")
print(f"Items only in mikro (no inventory count): {len(mikro_only_codes)}")
print()

# Show comparison for items with sayım data
with_sayim = verify[verify["Sayim_Depo"].notna()].sort_values("_key")
exact_match = (with_sayim["Diff"] == 0).sum()
close_match = (with_sayim["Diff"].abs() <= 2).sum()

print(f"Exact matches (Diff=0): {exact_match} / {len(with_sayim)}")
print(f"Close matches (|Diff|<=2): {close_match} / {len(with_sayim)}")
print()

# Show items with discrepancies
discrepancies = with_sayim[with_sayim["Diff"] != 0].copy()
print(f"Items with discrepancies ({len(discrepancies)}):")
print(discrepancies.to_string(index=False))
print()
print("Note: Positive Diff = lots have more than sayım; Negative = sayım has more")

Strategy defined. Proceeding to build item_definitions and lots tables.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BUILD ITEM DEFINITIONS TABLE
# ══════════════════════════════════════════════════════════════════════
# Get min/critical/max stock levels from sayım, item info from both sources

# Items from sayım (has inventory levels)
items_sayim = df_sayim.drop_duplicates("_key").copy()

# Get supplier from mikro (first occurrence per item)
supplier_map = lot_agg.drop_duplicates("_key", keep="first").set_index("_key")["Supplier"]

df_items_sayim = pd.DataFrame({
    "Malzeme Kodu":                  items_sayim["_key"].values,
    "Malzeme Adı":                   items_sayim["Malzeme Adı"].values,
    "Kategori":                      np.nan,
    "Departman":                     items_sayim["department"].values,
    "Birim":                         items_sayim["Birim"].values,
    "Min Stok":                      items_sayim["Minimum Stok Seviyesi"].values,
    "İdeal Stok Seviyesi (3 aylık)": items_sayim["Kritik Stok Seviyesi\n(3 aylık)"].values,
    "Maksimum Stok Seviyesi":        items_sayim["Maksimum Stok Seviyesi"].values,
    "Marka":                         items_sayim["Marka"].values,
    "Tedarikçi":                     items_sayim["_key"].map(supplier_map).values,
})

# Items only in mikro (no sayım data)
mikro_only_items = lot_agg[lot_agg["_key"].isin(mikro_only_codes)].drop_duplicates("_key", keep="first")

df_items_mikro = pd.DataFrame({
    "Malzeme Kodu":                  mikro_only_items["_key"].values,
    "Malzeme Adı":                   mikro_only_items["Item_Name"].values,
    "Kategori":                      np.nan,
    "Departman":                     mikro_only_items["Department"].values,
    "Birim":                         np.nan,
    "Min Stok":                      np.nan,
    "İdeal Stok Seviyesi (3 aylık)": np.nan,
    "Maksimum Stok Seviyesi":        np.nan,
    "Marka":                         np.nan,
    "Tedarikçi":                     mikro_only_items["Supplier"].values,
})

# Combine
df_item_definitions = pd.concat([df_items_sayim, df_items_mikro], ignore_index=True)
print(f"item_definitions: {len(df_item_definitions)} unique items")
print(f"  - from sayım:      {len(df_items_sayim)}")
print(f"  - from mikro-only: {len(df_items_mikro)}")

item_definitions: 90 unique items
  - from sayım:      70
  - from mikro-only: 20


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BUILD LOTS TABLE
# ══════════════════════════════════════════════════════════════════════
# Active lots: currentQuantity = Total_Received
# Depleted lots: currentQuantity = 0

lot_rows = []

# Lots from mikro (both active and depleted)
for _, row in lot_agg.iterrows():
    lot_rows.append({
        "Malzeme Kodu": row["_key"],
        "Lot No": row["Lot_No"],
        "Son Kullanma": row["Expiry"],
        "Geliş Tarihi": row["Received_Date"],
        "initialQuantity": int(row["Total_Received"]),
        "currentQuantity": int(row["currentQuantity"]),
        "Status": row["Status"],
        "Tedarikçi": row["Supplier"],
    })

# Items only in sayım (no lot data) → create a LEGACY-STOK lot
for _, row in items_sayim[items_sayim["_key"].isin(sayim_only_codes)].iterrows():
    depo = row[" Depo"] if pd.notna(row[" Depo"]) else 0
    expiry_raw = row.get("Son Kullanma Tarihi", np.nan)
    
    lot_rows.append({
        "Malzeme Kodu": row["_key"],
        "Lot No": "LEGACY-STOK",
        "Son Kullanma": expiry_raw if pd.notna(expiry_raw) and str(expiry_raw).strip().lower() != "yok" else np.nan,
        "Geliş Tarihi": np.nan,
        "initialQuantity": int(depo),
        "currentQuantity": int(depo),
        "Status": "ACTIVE" if depo > 0 else "DEPLETED",
        "Tedarikçi": np.nan,
    })

df_lots = pd.DataFrame(lot_rows)

# Stats
active_count = (df_lots["Status"] == "ACTIVE").sum()
depleted_count = (df_lots["Status"] == "DEPLETED").sum()
legacy_count = (df_lots["Lot No"] == "LEGACY-STOK").sum()

print(f"lots table: {len(df_lots)} total rows")
print(f"  ACTIVE: {active_count}")
print(f"  DEPLETED: {depleted_count}")
print(f"  LEGACY-STOK (sayım-only items): {legacy_count}")

lots table: 214 total rows
  - LEGACY-STOK (carries real count): 70
  - HISTORICAL  (qty=0, for history): 144


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FINAL VERIFICATION: Check totals match
# ══════════════════════════════════════════════════════════════════════

# Sum currentQuantity per item from df_lots
final_stock = df_lots.groupby("Malzeme Kodu")["currentQuantity"].sum().reset_index()
final_stock.columns = ["Malzeme Kodu", "App_Total_Stock"]

# Merge with sayım Depo
final_verify = final_stock.merge(
    df_sayim[["_key", " Depo"]].drop_duplicates("_key").rename(columns={"_key": "Malzeme Kodu", " Depo": "Sayim_Depo"}),
    on="Malzeme Kodu", how="left"
)
final_verify["Diff"] = final_verify["App_Total_Stock"] - final_verify["Sayim_Depo"].fillna(0)

# Add discrepancy info to item_definitions for export
df_item_definitions = df_item_definitions.merge(
    final_verify[["Malzeme Kodu", "App_Total_Stock", "Sayim_Depo", "Diff"]],
    on="Malzeme Kodu", how="left"
)

print("═" * 70)
print("FINAL VERIFICATION")
print("═" * 70)

with_sayim = final_verify[final_verify["Sayim_Depo"].notna()]
exact = (with_sayim["Diff"] == 0).sum()
print(f"\nItems with sayım data: {len(with_sayim)}")
print(f"Exact matches: {exact}")
print(f"Items with discrepancy: {len(with_sayim) - exact}")
print()

# Show discrepancies for manual review
disc = with_sayim[with_sayim["Diff"] != 0].sort_values("Diff")
if len(disc) > 0:
    print("Discrepancies (need manual review):")
    print(disc.to_string(index=False))
else:
    print("✓ All items match perfectly!")

✓ VERIFICATION PASSED: All items match — no double-counting.
  Checked 70 items with sayım data.

  Mikro-only items (no known stock): 14 — all have total=0: True


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DATA QUALITY REPORT
# ══════════════════════════════════════════════════════════════════════

print("═" * 70)
print("DATA QUALITY REPORT")
print("═" * 70)

# 1. Missing values in item_definitions
ITEM_COLS = ["Malzeme Kodu", "Malzeme Adı", "Kategori", "Departman", "Birim",
             "Min Stok", "İdeal Stok Seviyesi (3 aylık)", "Maksimum Stok Seviyesi", "Marka", "Tedarikçi"]

print("\n── item_definitions: Missing values per column ──")
for col in ITEM_COLS:
    if col in df_item_definitions.columns:
        missing = df_item_definitions[col].isna().sum()
        if missing > 0:
            print(f"  {col:<40s} {missing:>4d} / {len(df_item_definitions)}")

# 2. Lot status breakdown
print(f"\n── lots table breakdown ──")
print(f"  Total lots: {len(df_lots)}")
print(f"  ACTIVE: {active_count}")
print(f"  DEPLETED: {depleted_count}")
print(f"  LEGACY-STOK (sayım-only items): {legacy_count}")

# 3. Items with zero stock
zero_stock_items = df_item_definitions[df_item_definitions["App_Total_Stock"] == 0]
print(f"\n── Items with zero stock: {len(zero_stock_items)} ──")

# 4. Sayım-only items
print(f"\n── {len(sayim_only_codes)} items in sayım with NO lot history in mikro ──")
print("  These have a LEGACY-STOK lot created.")

# 5. Mikro-only items
print(f"\n── {len(mikro_only_codes)} items in mikro with NO inventory data in sayım ──")
print("  These have lots but no min/critical/max stock levels.")

# 6. Kategori is always blank
print(f"\n── Kategori column ──")
print("  ⚠ Kategori is blank for ALL items — fill manually.")

print("\n" + "═" * 70)
print("END OF REPORT")
print("═" * 70)

DATA QUALITY REPORT

── item_definitions: Missing values per column ──
  Kategori                                   90 / 90
  Departman                                   1 / 90
  Birim                                      20 / 90
  Min Stok                                   20 / 90
  İdeal Stok Seviyesi (3 aylık)              20 / 90
  Maksimum Stok Seviyesi                     20 / 90
  Marka                                      21 / 90
  Tedarikçi                                  35 / 90

── lots table breakdown ──
  LEGACY-STOK lots (carry actual stock): 70
  HISTORICAL lots  (qty=0, for history): 144

── Items with Depo = 0 (out of stock): 3 ──
   _key                                     Malzeme Adı
 AE2022 Magnesia 16 Viral DNA/RNA Extraction Kit (96rx)
 705001                                      Petri dish
HP7S00X                          Anyplex HPV28 100 test

── 29 items in sayım with NO order history in mikro ──
  These will only have a LEGACY-STOK lot, no historical lots.
 

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# EXPORT TO EXCEL
# ══════════════════════════════════════════════════════════════════════

output_path = CONVERTS_DIR / "mikro_migration_output.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    # Sheet 1: item_definitions (includes verification columns)
    df_item_definitions.to_excel(writer, sheet_name="item_definitions", index=False)
    
    # Sheet 2: lots
    df_lots.to_excel(writer, sheet_name="lots", index=False)
    
    # Sheet 3: discrepancies for manual review
    disc_df = df_item_definitions[df_item_definitions["Diff"] != 0][
        ["Malzeme Kodu", "Malzeme Adı", "App_Total_Stock", "Sayim_Depo", "Diff"]
    ].sort_values("Diff")
    disc_df.to_excel(writer, sheet_name="discrepancies", index=False)
    
    # Sheet 4: flat view (lots + item info merged)
    df_flat = df_lots.merge(
        df_item_definitions[["Malzeme Kodu", "Malzeme Adı", "Birim", "Min Stok", 
                             "İdeal Stok Seviyesi (3 aylık)", "Maksimum Stok Seviyesi", "Marka"]],
        on="Malzeme Kodu", how="left"
    )
    df_flat.to_excel(writer, sheet_name="flat_review", index=False)

print(f"✓ Saved to: {output_path}")
print(f"  Sheet 'item_definitions'  → {len(df_item_definitions)} items (with App_Total_Stock, Sayim_Depo, Diff)")
print(f"  Sheet 'lots'              → {len(df_lots)} lot rows")
print(f"  Sheet 'discrepancies'     → {len(disc_df)} items needing manual review")
print(f"  Sheet 'flat_review'       → combined view for review")

✓ Saved to: C:\Users\STREAM\Desktop\order tracking\converts\mikro_migration_output.xlsx
  Sheet 'item_definitions'  → 90 unique items
  Sheet 'lots'              → 214 lot rows (70 LEGACY + 144 HISTORICAL)
  Sheet 'flat_review'       → combined flat view for human review


In [ ]:
# (cleanup - remove old exploratory code)